# Seg + SD3 Inpaint (Reproducible Notebook)

This notebook reproduces the web demo pipeline: **EfficientSAM segmentation -> SD3 inpainting**.

Input image is resolved as `img/xai506_example_image.jpg` from repository root.

If model download is gated in your environment, run `huggingface-cli login` first and accept model terms on Hugging Face.


In [ ]:
# Install environment (Colab/Jupyter style)
!pip install -q torch torchvision diffusers==0.31.0 transformers pillow huggingface_hub accelerate matplotlib numpy


In [ ]:
import importlib.util
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageFilter
from huggingface_hub import hf_hub_download
from torchvision import transforms
from torchvision.transforms import ToTensor

REPO_ID = 'merve/EfficientSAM'
SD3_INPAINT_MODEL_ID = 'IrohXu/stable-diffusion-3-inpainting'
SD3_BASE_MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
IMAGE_REL_PATH = Path('img') / 'xai506_example_image.jpg'

def pick_best_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device('cpu')

    best_idx = 0
    best_free = -1
    for idx in range(torch.cuda.device_count()):
        try:
            free_bytes, _ = torch.cuda.mem_get_info(idx)
        except Exception:
            free_bytes = 0
        if free_bytes > best_free:
            best_free = free_bytes
            best_idx = idx
    return torch.device(f'cuda:{best_idx}')

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / IMAGE_REL_PATH).exists():
            return p
    raise FileNotFoundError(f'Cannot find {IMAGE_REL_PATH} from {start}')

repo_root = find_repo_root(Path.cwd())
image_path = repo_root / IMAGE_REL_PATH
device = pick_best_device()
dtype = torch.float16 if device.type == 'cuda' else torch.float32

print('repo_root =', repo_root)
print('image_path =', image_path)
print('device =', device, '| dtype =', dtype)

image = Image.open(image_path).convert('RGB')
plt.figure(figsize=(6, 4))
plt.imshow(image)
plt.title('Input image')
plt.axis('off')
plt.show()


In [ ]:
def load_efficientsam(preferred_device: torch.device):
    runtime_device = preferred_device

    if preferred_device.type == 'cuda':
        try:
            ckpt = hf_hub_download(repo_id=REPO_ID, filename='efficient_sam_s_gpu.jit')
            model = torch.jit.load(ckpt, map_location=str(preferred_device))
            model.eval()
            return model, runtime_device, 'efficient_sam_s_gpu.jit'
        except Exception as e:
            print('GPU checkpoint load failed; fallback to CPU checkpoint:', e)
            runtime_device = torch.device('cpu')

    ckpt = hf_hub_download(repo_id=REPO_ID, filename='efficient_sam_s_cpu.jit')
    model = torch.jit.load(ckpt, map_location='cpu')
    model.eval()
    return model, runtime_device, 'efficient_sam_s_cpu.jit'

def resize_longest_side(img: Image.Image, target: int = 1024):
    w, h = img.size
    scale = target / max(w, h)
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    return img.resize((nw, nh)), scale

def make_overlay(img: Image.Image, mask_bool: np.ndarray, alpha: float = 0.45):
    base = np.array(img.convert('RGB'), dtype=np.float32)
    color = np.array([255.0, 0.0, 0.0], dtype=np.float32)
    out = base.copy()
    out[mask_bool] = (1.0 - alpha) * out[mask_bool] + alpha * color
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))

def segment_with_points(img: Image.Image, points, point_labels, model, runtime_device, input_size=1024):
    resized, scale = resize_longest_side(img, input_size)
    scaled_points = np.array([[int(round(x * scale)), int(round(y * scale))] for x, y in points], dtype=np.float32)
    labels_np = np.array(point_labels, dtype=np.int64)

    img_tensor = ToTensor()(np.array(resized)).unsqueeze(0).to(runtime_device)
    pts_tensor = torch.tensor(scaled_points, dtype=torch.float32, device=runtime_device).view(1, 1, -1, 2)
    labels_tensor = torch.tensor(labels_np, dtype=torch.int64, device=runtime_device).view(1, 1, -1)

    with torch.no_grad():
        logits, iou = model(img_tensor, pts_tensor, labels_tensor)

    masks = (torch.sigmoid(logits[0, 0]) > 0.5).cpu().numpy()
    ious = iou[0, 0].detach().cpu().numpy()
    best_idx = int(np.argmax(ious))
    best_mask_small = (masks[best_idx].astype(np.uint8) * 255)

    mask_full = Image.fromarray(best_mask_small, mode='L').resize(img.size, resample=Image.NEAREST)
    mask_bool = np.array(mask_full) > 127
    overlay = make_overlay(img, mask_bool, alpha=0.45)

    return {
        'mask_image': mask_full,
        'mask_bool': mask_bool,
        'overlay': overlay,
        'ious': ious,
        'best_idx': best_idx,
    }

sam_model, sam_device, sam_ckpt = load_efficientsam(device)
print('EfficientSAM loaded:', sam_ckpt, '| runtime device =', sam_device)


In [ ]:
# Prompt points: edit these coordinates for your target object
# label 1 = foreground, label 0 = background
points = [[image.width // 2, image.height // 2]]
point_labels = [1]

seg_result = segment_with_points(
    img=image,
    points=points,
    point_labels=point_labels,
    model=sam_model,
    runtime_device=sam_device,
    input_size=1024,
)

print('Predicted IoUs:', seg_result['ious'])
print('Selected mask index:', seg_result['best_idx'])

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title('Input')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(seg_result['overlay'])
plt.title('Segmentation overlay')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(seg_result['mask_image'], cmap='gray')
plt.title('Auto mask')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
from diffusers import DiffusionPipeline

def load_sd3_inpaint_pipeline(runtime_device: torch.device, runtime_dtype: torch.dtype):
    pipeline_file = hf_hub_download(
        repo_id=SD3_INPAINT_MODEL_ID,
        filename='pipeline_stable_diffusion_3_inpaint.py',
    )

    with open(pipeline_file, 'r', encoding='utf-8') as f:
        source = f.read()

    buggy = 'latent_timestep = timesteps[:1].repeat(batch_size * num_inference_steps)'
    fixed = 'latent_timestep = timesteps[:1].repeat(batch_size * num_images_per_prompt)'
    if buggy in source:
        source = source.replace(buggy, fixed)

    patched_file = Path(tempfile.gettempdir()) / 'pipeline_stable_diffusion_3_inpaint_patched.py'
    with open(patched_file, 'w', encoding='utf-8') as f:
        f.write(source)

    spec = importlib.util.spec_from_file_location('custom_sd3_inpaint_pipeline', str(patched_file))
    if spec is None or spec.loader is None:
        raise RuntimeError('Failed to load custom pipeline module')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    pipeline_cls = getattr(mod, 'StableDiffusion3InpaintPipeline', None)
    if pipeline_cls is None or not issubclass(pipeline_cls, DiffusionPipeline):
        raise RuntimeError('Invalid custom SD3 inpaint pipeline class')

    pipe = pipeline_cls.from_pretrained(SD3_BASE_MODEL_ID, torch_dtype=runtime_dtype).to(runtime_device)
    if hasattr(pipe, 'enable_attention_slicing'):
        pipe.enable_attention_slicing()
    return pipe

def resize_for_max_side(img: Image.Image, mask: Image.Image, max_side: int = 1024):
    w, h = img.size
    if max(w, h) <= int(max_side):
        return img, mask

    scale = float(max_side) / float(max(w, h))
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    return (
        img.resize((nw, nh), Image.LANCZOS),
        mask.resize((nw, nh), Image.NEAREST),
    )

def center_crop_multiple_of_64(img: Image.Image):
    w, h = img.size
    nw = max(64, (w // 64) * 64)
    nh = max(64, (h // 64) * 64)

    left = (w - nw) // 2
    top = (h - nh) // 2
    right = left + nw
    bottom = top + nh

    return img.crop((left, top, right, bottom)), (left, top, right, bottom)

sd3_pipe = load_sd3_inpaint_pipeline(device, dtype)
print('SD3 inpaint pipeline loaded on', device)


In [ ]:
def run_sd3_inpaint(
    img: Image.Image,
    mask_img: Image.Image,
    prompt: str,
    negative_prompt: str = '',
    num_inference_steps: int = 30,
    guidance_scale: float = 7.0,
    strength: float = 0.6,
    seed: int = -1,
    max_side: int = 1024,
    mask_expand_px: int = 12,
):
    work_image = img.convert('RGB')
    work_mask = mask_img.convert('L')

    if int(mask_expand_px) > 0:
        k = int(mask_expand_px) * 2 + 1
        k = max(3, min(k, 255))
        work_mask = work_mask.filter(ImageFilter.MaxFilter(size=k))

    work_image, work_mask = resize_for_max_side(work_image, work_mask, max_side=max_side)

    original_canvas = work_image.copy()
    image_c, crop_box = center_crop_multiple_of_64(work_image)
    mask_c, _ = center_crop_multiple_of_64(work_mask)

    image_t = transforms.ToTensor()(image_c).unsqueeze(0).to(device=device, dtype=dtype)
    mask_t = transforms.ToTensor()(mask_c).to(device=device, dtype=dtype)

    generator = None
    if int(seed) >= 0:
        generator = torch.Generator(device=str(device)).manual_seed(int(seed))

    out = sd3_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt.strip() or None,
        image=image_t,
        mask_image=mask_t,
        height=image_t.shape[-2],
        width=image_t.shape[-1],
        num_inference_steps=max(1, int(num_inference_steps)),
        guidance_scale=float(guidance_scale),
        strength=float(strength),
        generator=generator,
    ).images[0]

    left, top, right, bottom = crop_box
    if out.size != (right - left, bottom - top):
        out = out.resize((right - left, bottom - top), resample=Image.LANCZOS)

    original_canvas.paste(out, (left, top))
    return original_canvas, work_mask

prompt = 'Turn the segmented object into a glossy red ceramic sculpture.'
negative_prompt = 'blurry, low quality, distorted, artifacts'

inpaint_result, expanded_mask = run_sd3_inpaint(
    img=image,
    mask_img=seg_result['mask_image'],
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.0,
    strength=0.6,
    seed=-1,
    max_side=1024,
    mask_expand_px=12,
)

plt.figure(figsize=(14, 8))
plt.subplot(2, 2, 1)
plt.imshow(image)
plt.title('Input')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(seg_result['overlay'])
plt.title('Segmentation overlay')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(expanded_mask, cmap='gray')
plt.title('Expanded mask used for inpaint')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(inpaint_result)
plt.title('SD3 inpaint result')
plt.axis('off')

plt.tight_layout()
plt.show()


## Notes
- First run is slow because models are downloaded and loaded.
- If output changes too much, lower `strength` (for example 0.4).
- If seams appear around the object, tune `mask_expand_px` (for example 8~20).
